In [1]:
!pip uninstall -y peft
!pip install "transformers==4.41.2" "sentence-transformers==3.0.1"
!pip install -U datasets

Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 110.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      S

# 零、总述

文本分类是 NLP 中的一项常见任务。从情感分析和意图识别，到实体提取和语言检测，文本分类被广泛应用。表示模型（Representation Models）和生成模型（Generative Models）在文本分类中的重要作用不容忽视。

**表示模型（Representation Model）** 就是把文本、图片、音频等原始数据，转换成计算机更容易处理的 **向量表示**，例如:
$$
\text{[I Like Apple]} \xrightarrow{表示成} \mathbf{v} \in \mathbb{R}^d
$$

这样，向量 $\mathbf{v}$ 就可以综合表示
- 词语含义
- 上下文信息
- 语法关系
- 情感倾向
- 主题信息
- 实体关系
- 等等

# 一、影评的情感分析

本章我们将使用著名的 rotten_tomatoes 数据集来评估我们的模型

rotten_tomatoes 是一个收集了大量不同的影评的数据集，该数据集中包含来自 Rotten Tomatoes 网站的 5331 条正面和 5531 条负面的电影评论

In [2]:
from datasets import load_dataset

data = load_dataset("rotten_tomatoes")
data

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

可以看到此数据集的训练集是8530条数据，验证集和测试集均为1066条数据

In [3]:
train_sets = data["train"]
test_sets = data["test"]

In [4]:
data["train"][0, -1]

{'text': ['the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .',
  'things really get weird , though not particularly scary : the movie is all portent and no content .'],
 'label': [1, 0]}

In [5]:
train_sets.shape

(8530, 2)

# 二、使用表示模型进行文本分类

使用预训练的表示模型进行分类，通常有两种方式：一是使用**特定任务模型**，二是使用**嵌入模型**，这些模型是通过特定下游任务上微调基础模型（如 BERT）而创建的。

BERT 发展了许多的变体，按时间顺序先后出现的顺序为：BERT -> RoBERTa -> DistilBERT -> ALBERT -> DeBERTa

下面我们来分别介绍一下这几种网络：

1. BERT

BERT(全称：Bidirectional Encoder Representations from Transformers)，它的核心特点是可以让**一个中心词看到它左右两边的上下文**，另一个特点是*使用了掩码的机制*

2. RoBERTa

RoBERTa(全称：Robustly Optimized BERT Pretraining Approach)，它没有大改 BERT 的网络架构，而是发现*BERT本省其实还没有被训练充分*, 所以主要**优化了训练方式**，例如：
- 使用更多的训练数据
- 增加训练的时长
- 增加每次训练的批次数据大小
- 使用动态掩码机制
- 等等

3. DistilBERT

DistilBERT 的采用的是 **Knowledge Distillation**,即：知识蒸馏的方法。通俗的来讲就是将大模型 BERT 作为 Teacher, 训练一个更小的 Student, 之后让这个 Student 去模仿 Teacher 的行为，从而达到相似的效果

4. ALBERT

ALBERT（全称：A Lite BERT）,它和模型蒸馏一样，解决的也是BERT参数太多的问题，但是它直接修改的是模型的参数设计，有两个比较重要的思想：
- Factorized Embedding Parameterization：将巨大的词嵌入矩阵进行分解，减少参数
- Cross-layer Parameter Sharing: 不同的 Transformer Layer 共享参数

5. DeBERTa

DeBERTa(全称：Decoding-enhanced BERT with Disentangled Attention),它对 BERT 的 Attention 机制进行了更深层的修改，最重要的思想是**解耦注意力（Disentangled Attention）**。通常情况下，普通的 BERT 会把 Token 的内容信息 + 位置信息融合在一块，而这会将这两种信息拆开来考虑

## 2.1 使用特定任务模型

我们使用的 `cardiffnlp/twitter-roberta-base-sentiment-latest` 是 Twitter 针对情感分析进行微调的一个 RoBERTa 的模型

In [6]:
from transformers import pipeline

model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"

# 使用 pipeline
pipe = pipeline(
    model=model_path,
    tokenizer=model_path,
    return_all_scores=True,
    device="cuda:0"
)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/pipelines/text_classification.py:104: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


这里介绍一下 `tqdm`， 它是 Python 常用的一个进度条库，它的名字来自阿拉伯语 `taqaddum`，最核心的作用是：**给循环、迭代器、数据处理任务增加一个实时的进度条**

In [7]:
import numpy as np
from tqdm import tqdm
from transformers.pipelines.pt_utils import KeyDataset

y_pred = []
for output in tqdm(pipe(KeyDataset(data["test"], "text")), total=len(data["test"])):
    negative_score = output[0]["score"]
    positive_score = output[2]["score"]
    assignment = np.argmax([negative_score, positive_score])
    y_pred.append(assignment)


100%|██████████| 1066/1066 [00:21<00:00, 48.58it/s]


In [8]:
from sklearn.metrics import classification_report

# 我们使用 sklearn 自带的一个 classification_report 来展示分类报告
def evaluate_performance(y_true, y_pred):
    performance = classification_report(
        y_true, y_pred,
        target_names=["差评", "好评"]
    )
    print(performance)

In [9]:
evaluate_performance(test_sets["label"], y_pred)

              precision    recall  f1-score   support

          差评       0.76      0.88      0.81       533
          好评       0.86      0.72      0.78       533

    accuracy                           0.80      1066
   macro avg       0.81      0.80      0.80      1066
weighted avg       0.81      0.80      0.80      1066



## 2.2 利用嵌入向量进行分类任务

### 2.2.1 有监督分类

In [10]:
from sentence_transformers import SentenceTransformer

# 加载模型
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

# 将文本转换成嵌入向量
train_embeddings = model.encode(data["train"]["text"], show_progress_bar=True)
test_embeddings = model.encode(data["test"]["text"], show_progress_bar=True)
print(train_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/267 [00:00<?, ?it/s]

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

(8530, 768)


In [11]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(random_state=42)
clf.fit(train_embeddings, data["train"]["label"])

LogisticRegression(random_state=42)

In [12]:
y_pred = clf.predict(test_embeddings)
evaluate_performance(test_sets["label"], y_pred)

              precision    recall  f1-score   support

          差评       0.85      0.86      0.85       533
          好评       0.86      0.85      0.85       533

    accuracy                           0.85      1066
   macro avg       0.85      0.85      0.85      1066
weighted avg       0.85      0.85      0.85      1066



### 2.2.2 无监督分类
我们可以平均每个类的嵌入向量并使用 Cosine Similarity 来预测哪个类别更匹配文档

In [13]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.metrics.pairwise import cosine_similarity

# 将训练集中属于同一标签的文档向量取平均，得到每个标签对应的"类别中心向量"
df = pd.DataFrame(np.hstack([train_embeddings, np.array(train_sets["label"]).reshape(-1, 1)]))

# 这里的 `768` 代表 DataFrame 的第 768 列(标签，label⚠️)，也就是标签所在的列
# 另外，需要注意的是⚠️⚠️⚠️: `768` 传递的 groupby 函数的 by 参数里面的 label ⚠️⚠️⚠️
avg_target_embeddings = df.groupby(768).mean().values


# 预测
sim_matrix = cosine_similarity(test_embeddings, avg_target_embeddings)
y_pred = np.argmax(sim_matrix, axis=1)

evaluate_performance(test_sets["label"], y_pred)

              precision    recall  f1-score   support

          差评       0.85      0.84      0.84       533
          好评       0.84      0.85      0.84       533

    accuracy                           0.84      1066
   macro avg       0.84      0.84      0.84      1066
weighted avg       0.84      0.84      0.84      1066



### 2.2.3 Zero-shot Classification

一般中文翻译为：**零样本分类**，它的核心思想是：**模型没有针对具体分类任务进行专门训练，也可以直接完成分类，**可以直接根据“类别名称”完成分类；

比如此节中说的影评，将好评（A positive review）和差评（A negative review）作为向量，计算它们与每个类别的平均嵌入向量的余弦相似度，最后去做预测

In [14]:
label_embeddings = model.encode(["A negative review", "A positive review"])

sim_matrix = cosine_similarity(test_embeddings, avg_target_embeddings)
y_pred = np.argmax(sim_matrix, axis=1)

evaluate_performance(test_sets["label"], y_pred)

              precision    recall  f1-score   support

          差评       0.85      0.84      0.84       533
          好评       0.84      0.85      0.84       533

    accuracy                           0.84      1066
   macro avg       0.84      0.84      0.84      1066
weighted avg       0.84      0.84      0.84      1066



## 2.3 使用生成模型进行文本分类

In [15]:
pipe = pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    device="cuda:0"
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [16]:
prompt = "Is the following sentence positive or negative? "
data = data.map(lambda example: {"t5": prompt + example['text']})
data

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 't5'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label', 't5'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label', 't5'],
        num_rows: 1066
    })
})

In [17]:
y_pred = []
for output in tqdm(pipe(KeyDataset(data["test"], "t5")), total=len(data["test"])):
    text = output[0]["generated_text"]
    y_pred.append(0 if text == "negative" else 1)

  0%|          | 0/1066 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1168: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
100%|██████████| 1066/1066 [00:41<00:00, 25.83it/s]


In [18]:
evaluate_performance(data["test"]["label"], y_pred)

              precision    recall  f1-score   support

          差评       0.83      0.85      0.84       533
          好评       0.85      0.83      0.84       533

    accuracy                           0.84      1066
   macro avg       0.84      0.84      0.84      1066
weighted avg       0.84      0.84      0.84      1066

